# Experiment 03 — Feature Ablation

**Goal:** Determine whether exogenous weather features improve STLF
compared to a univariate (load-only) model.

| Experiment | Features |
|------------|----------|
| A (Univariate) | Load only |
| B (Multivariate) | Load + temperature, rain, humidity, sunshine |

Everything else is held constant: model, split, lookback, horizon,
optimizer, epochs, early stopping, seed.

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────
import platform, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import tensorflow as tf
import sklearn

from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import chronological_split, fit_preprocessor, transform_data, inverse_y
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.bilstm import build_bilstm
from src.training.trainer import train_model
from src.evaluation.point_metrics import compute_all_metrics, horizon_wise_metrics, compute_naive_baselines

print(f"Python: {platform.python_version()} | TF: {tf.__version__} | NumPy: {np.__version__}")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
LOOKBACK   = config["windowing"]["lookback"]
HORIZON    = config["windowing"]["horizon"]
TARGET_COL = config["data"]["target_col"]
WEATHER    = config["data"]["weather_features"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]

# Data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    for fb in [PROJECT_ROOT / config["data"]["path"], PROJECT_ROOT / "df_combined_AT.csv"]:
        if fb.exists():
            DATA_PATH = fb
            break
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "deterministic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATA_PATH}")

In [ ]:
# ── Cell 4: Load & Split ─────────────────────────────────────────
df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
train_df, val_df, test_df = chronological_split(
    df, config["split"]["train_ratio"], config["split"]["val_ratio"]
)
full_series_mw = df[TARGET_COL].values
test_start_idx = len(train_df) + len(val_df)

In [ ]:
# ── Cell 5: Run Ablation ─────────────────────────────────────────
experiments = {
    "univariate": {
        "feature_cols": [TARGET_COL],
        "use_yj": False,
    },
    "multivariate": {
        "feature_cols": [TARGET_COL] + WEATHER,
        "use_yj": config["preprocessing"]["use_yeojohnson"],
    },
}

all_results = {}

for exp_name, exp_cfg in experiments.items():
    print(f"\n{'='*60}")
    print(f"  Experiment: {exp_name}")
    print(f"  Features: {exp_cfg['feature_cols']}")
    print(f"{'='*60}")
    
    # Reset seed for fair comparison
    set_seed(SEED)
    
    # Preprocessing
    skewed = config["preprocessing"]["skewed_cols"] if exp_cfg["use_yj"] else None
    preprocessor = fit_preprocessor(
        train_df, TARGET_COL, exp_cfg["feature_cols"],
        skewed_cols=skewed, use_yeojohnson=exp_cfg["use_yj"]
    )
    
    X_tr, y_tr = transform_data(train_df, preprocessor)
    X_va, y_va = transform_data(val_df, preprocessor)
    X_te, y_te = transform_data(test_df, preprocessor)
    
    # Windowing
    Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, LOOKBACK, HORIZON)
    Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, LOOKBACK, HORIZON)
    Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, LOOKBACK, HORIZON)
    
    # Build & train model
    mdl = build_bilstm(LOOKBACK, Xw_tr.shape[2], HORIZON, UNITS, DROPOUT, LR)
    tr_res = train_model(mdl, Xw_tr, yw_tr, Xw_va, yw_va, EPOCHS, BATCH_SIZE, PATIENCE, verbose=1)
    
    # Predict & inverse transform
    pred_scaled = mdl.predict(Xw_te)
    pred_mw = inverse_y(pred_scaled, preprocessor)
    actual_mw = inverse_y(yw_te, preprocessor)
    
    # Metrics
    mets = compute_all_metrics(actual_mw, pred_mw)
    hw = horizon_wise_metrics(actual_mw, pred_mw)
    
    all_results[exp_name] = {
        "metrics": mets,
        "horizon_mape": hw["mape"],
        "best_epoch": tr_res["best_epoch"],
        "best_val_loss": tr_res["best_val_loss"],
        "history": tr_res["history"],
    }
    
    print(f"\n  Results for {exp_name}:")
    for k, v in mets.items():
        print(f"    {k}: {v:.4f}")

In [ ]:
# ── Cell 6: Comparison Table ─────────────────────────────────────
print("\n" + "=" * 60)
print("FEATURE ABLATION COMPARISON")
print("=" * 60)
print(f"{'Experiment':<15s} {'MAE':>10s} {'RMSE':>10s} {'MAPE':>10s} {'sMAPE':>10s}")
print("-" * 57)
for name, res in all_results.items():
    m = res["metrics"]
    print(f"{name:<15s} {m['MAE']:>10.2f} {m['RMSE']:>10.2f} {m['MAPE']:>10.2f} {m['sMAPE']:>10.2f}")

In [ ]:
# ── Cell 7: Visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
metric_names = ["MAE", "RMSE", "MAPE", "sMAPE"]
x = np.arange(len(metric_names))
width = 0.35

uni_vals = [all_results["univariate"]["metrics"][m] for m in metric_names]
multi_vals = [all_results["multivariate"]["metrics"][m] for m in metric_names]

axes[0].bar(x - width/2, uni_vals, width, label="Univariate")
axes[0].bar(x + width/2, multi_vals, width, label="Multivariate")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].set_title("Metrics Comparison")
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

# Horizon-wise MAPE
h_range = range(1, HORIZON + 1)
axes[1].plot(h_range, all_results["univariate"]["horizon_mape"], "o-", label="Univariate", markersize=3)
axes[1].plot(h_range, all_results["multivariate"]["horizon_mape"], "s-", label="Multivariate", markersize=3)
axes[1].set_title("MAPE by Forecast Horizon")
axes[1].set_xlabel("Horizon (h)")
axes[1].set_ylabel("MAPE (%)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "feature_ablation.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 8: Save Results ─────────────────────────────────────────
# Save individual results
for name, res in all_results.items():
    rec = {
        "experiment": "03_feature_ablation",
        "variant": name,
        "model": "BiLSTM",
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "seed": SEED,
        "best_epoch": res["best_epoch"],
        "best_val_loss": res["best_val_loss"],
        "test_metrics": res["metrics"],
    }
    tag = "uni" if name == "univariate" else "multi"
    out_path = RESULTS_DIR / f"bilstm_{tag}_{HORIZON}h_seed{SEED}.json"
    with open(out_path, "w") as f:
        json.dump(rec, f, indent=2)
    print(f"Saved: {out_path}")